In [1]:
import os
os.chdir('..')

from typing import Dict, Tuple
import numpy as np
import pandas as pd
from prettytable import PrettyTable
from src.helper_functions.data_loader import *
from src.helper_functions.data_splitter import chronological_split_per_user
from src.helper_functions.js_divergence import compute_group_js_divergence

### Compute basic stats for all datasets

In [2]:
def compute_gini_coefficient(interactions: pd.Series) -> float:
    """Compute the Gini coefficient for a given distribution."""
    sorted_values = np.sort(interactions)
    n = len(sorted_values)
    cum_values = np.cumsum(sorted_values)
    gini = (1 + 1/n - 2 * np.sum(cum_values / cum_values[-1]) / n)
    return gini

def compute_global_stats(df: pd.DataFrame, reference_df: pd.DataFrame = None) -> Tuple[int, int, int, float, float, float, float]:
    universe = df if reference_df is None else reference_df
    users_index = pd.Index(universe['User_id'].unique())
    items_index = pd.Index(universe['Item_id'].unique())
    n_users = len(users_index)
    n_items = len(items_index)
    interactions = df.shape[0]
    mean_item_degree = interactions / n_items
    sparsity = 1 - (interactions / (n_users * n_items))

    interactions_per_user = df.groupby('User_id').size().reindex(users_index, fill_value=0)
    interactions_per_item = df.groupby('Item_id').size().reindex(items_index, fill_value=0)
    gini_users = compute_gini_coefficient(interactions_per_user)
    gini_items = compute_gini_coefficient(interactions_per_item)
    
    return n_users, n_items, interactions, mean_item_degree, sparsity, gini_users, gini_items

def final_fitting_split(data: pd.DataFrame) -> pd.DataFrame:
    train_data, val_data, _ = chronological_split_per_user(data, return_dataframes=True)
    return pd.concat([train_data, val_data], ignore_index=True)

def compute_local_stats(df: pd.DataFrame, groups: Dict[str, list]) -> Dict[str, Dict[str, float]]:
    group_stats = {}
    for group, users in groups.items():
        group_df = df[df['User_id'].isin(users)]
        interactions = group_df.shape[0]
        count_users = len(users)
        avg_interactions = group_df.groupby('User_id').size().mean() # average interactions per user
        avg_rating = group_df.groupby('User_id')['Interaction'].mean().mean() # average rating per user
        avg_max_rating = group_df.groupby('User_id')['Interaction'].max().mean() # average max rating per user
        avg_min_rating = group_df.groupby('User_id')['Interaction'].min().mean()
        group_stats[group] = {
            "count": count_users,
            "interactions": interactions,
            "avg_interactions": round(avg_interactions),
            "avg_rating": round(avg_rating, 2),
            "avg_max_rating": round(avg_max_rating, 2),
            "avg_min_rating": round(avg_min_rating, 2)
        }
    return group_stats

def add_local_stats(table: PrettyTable, name: str, attribute: str, data: pd.DataFrame, 
                    groups: Dict[str, list], total_interactions: int):
    if groups is None:
        # Skip if there are no groups (e.g., no age groups for FTKY dataset)
        return
    if len(groups) < 2:
        return
    
    group_stats = compute_local_stats(data, groups)
    group_keys = list(groups.keys())
    group_js = compute_group_js_divergence(data, groups)
    
    # First group
    g1_count = group_stats.get(group_keys[0], {}).get('count', 0)
    g1_interactions = group_stats.get(group_keys[0], {}).get('interactions', 0)
    g1_avg_interactions = group_stats.get(group_keys[0], {}).get('avg_interactions', 0)
    g1_avg_rating = group_stats.get(group_keys[0], {}).get('avg_rating', 0)
    g1_avg_max_rating = group_stats.get(group_keys[0], {}).get('avg_max_rating', 0)
    g1_avg_min_rating = group_stats.get(group_keys[0], {}).get('avg_min_rating', 0)

    # Second group
    g2_count = group_stats.get(group_keys[1], {}).get('count', 0)
    g2_interactions = group_stats.get(group_keys[1], {}).get('interactions', 0)
    g2_avg_interactions = group_stats.get(group_keys[1], {}).get('avg_interactions', 0)
    g2_avg_rating = group_stats.get(group_keys[1], {}).get('avg_rating', 0)
    g2_avg_max_rating = group_stats.get(group_keys[1], {}).get('avg_max_rating', 0)
    g2_avg_min_rating = group_stats.get(group_keys[1], {}).get('avg_min_rating', 0)

    # Percentage with respect to total interactions
    g1_percentage = (g1_interactions / total_interactions * 100)
    g2_percentage = (g2_interactions / total_interactions * 100)

    table.add_row([
        name,
        attribute,
        f"{group_keys[0]}: {g1_count}",
        f"{g1_interactions} ({g1_percentage:.1f}%)",
        f"{g1_avg_interactions}",
        f"{g1_avg_rating:.2f}",
        f"{g1_avg_max_rating:.2f}",
        f"{g1_avg_min_rating:.2f}",
        f"{group_keys[1]}: {g2_count}",
        f"{g2_interactions} ({g2_percentage:.1f}%)",
        f"{g2_avg_interactions}",
        f"{g2_avg_rating:.2f}",
        f"{g2_avg_max_rating:.2f}",
        f"{g2_avg_min_rating:.2f}",
        f"{group_js:.3f}"
    ])

def display_stats():
    ml100k, groups_gender_ml100k, groups_age_ml100k = load_ml_100k()
    ml1m, groups_gender_ml1m, groups_age_ml1m = load_ml_1m()
    lfm1k, groups_gender_lfm1k, groups_age_lfm1k = load_lastfm_1k()
    fnyc, groups_gender_fnyc, _ = load_foursquare("NYC")
    ftky, groups_gender_ftky, _ = load_foursquare("TKY")

    table_global = PrettyTable()
    table_global.field_names = ["Dataset", "|U|", "|I|", "inter(U)", "Mean item degree", "Sparsity", "Gini (Users)", "Gini (Items)"]

    table_local = PrettyTable()
    table_local.field_names = [
        "Dataset", "Attribute", "|G1|", "total inter(G1) %", "avg inter(G1)", "avg rating(G1)", 
        "avg max rating(G1)", "avg min rating(G1)", "|G2|", "total inter(G2) %", "avg inter(G2)", 
        "avg rating(G2)", "avg max rating(G2)", "avg min rating(G2)", "JS divergence"
    ]

    datasets = {
        "ML100K": (ml100k, groups_gender_ml100k, groups_age_ml100k),
        "ML1M": (ml1m, groups_gender_ml1m, groups_age_ml1m),
        "LFM1K": (lfm1k, groups_gender_lfm1k, groups_age_lfm1k),
        "FTKY": (ftky, groups_gender_ftky, None),
        "FNYC": (fnyc, groups_gender_fnyc, None),
    }

    table_fit_global = PrettyTable()
    table_fit_global.field_names = ["Dataset", "|U|", "|I|", "inter(U)", "Mean item degree", "Sparsity", "Gini (Users)", "Gini (Items)"]

    table_fit_local = PrettyTable()
    table_fit_local.field_names = [
        "Dataset", "Attribute", "|G1|", "total inter(G1) %", "avg inter(G1)", "avg rating(G1)", 
        "avg max rating(G1)", "avg min rating(G1)", "|G2|", "total inter(G2) %", "avg inter(G2)", 
        "avg rating(G2)", "avg max rating(G2)", "avg min rating(G2)", "JS divergence"
    ]

    for name, (data, groups_gender, groups_age) in datasets.items():
        users, items, interactions, mean_item_degree, sparsity, gini_users, gini_items = compute_global_stats(data)
        table_global.add_row([name, users, items, interactions, f"{mean_item_degree:.1f}", f"{sparsity*100:.1f}%", f"{gini_users:.3f}", f"{gini_items:.3f}"])

        fit_data = final_fitting_split(data)
        fit_users, fit_items, fit_interactions, fit_mean_item_degree, fit_sparsity, fit_gini_users, fit_gini_items = compute_global_stats(fit_data, reference_df=data)
        table_fit_global.add_row([name, fit_users, fit_items, fit_interactions, f"{fit_mean_item_degree:.1f}", f"{fit_sparsity*100:.1f}%", f"{fit_gini_users:.3f}", f"{fit_gini_items:.3f}"])

        # Local stats for gender groups (m, f)
        add_local_stats(table_local, name, "Gender", data, groups_gender, interactions)
        add_local_stats(table_fit_local, name, "Gender", fit_data, groups_gender, fit_interactions)

        # Local stats for age groups (y, o)
        add_local_stats(table_local, name, "Age", data, groups_age, interactions)
        add_local_stats(table_fit_local, name, "Age", fit_data, groups_age, fit_interactions)

    print("Full preprocessed dataset")
    print(table_global)
    print(table_local)
    print("Chronological train+validation split used for final fitting")
    print(table_fit_global)
    print(table_fit_local)

display_stats()

Full preprocessed dataset
+---------+------+-------+----------+------------------+----------+--------------+--------------+
| Dataset | |U|  |  |I|  | inter(U) | Mean item degree | Sparsity | Gini (Users) | Gini (Items) |
+---------+------+-------+----------+------------------+----------+--------------+--------------+
|  ML100K | 943  |  1682 |  100000  |       59.5       |  93.7%   |    0.472     |    0.629     |
|   ML1M  | 6040 |  3706 | 1000209  |      269.9       |  95.5%   |    0.529     |    0.634     |
|  LFM1K  | 265  | 51359 |  198825  |       3.9        |  98.5%   |    0.441     |    0.638     |
|   FTKY  | 7240 |  5785 |  353847  |       61.2       |  99.2%   |    0.316     |    0.537     |
|   FNYC  | 4832 |  5651 |  182164  |       32.2       |  99.3%   |    0.268     |    0.467     |
+---------+------+-------+----------+------------------+----------+--------------+--------------+
+---------+-----------+---------+-------------------+---------------+----------------+------